# AI 07 - Ev tapşırığı (part 2)
# Avtomobilin qiymətinin proqnozlaşdırılması

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
FILE_PATH = 'Used_Cars_Homework.csv'
data = pd.read_csv(FILE_PATH)

# 1. İlkin təhlil (EDA)

In [ ]:
data.head(10)

In [ ]:
data.shape

In [ ]:
data.dtypes

In [ ]:
data.isna().sum()

In [ ]:
data.describe()

In [ ]:
print('Min qiymət:', data['price_azn'].min())
print('Max qiymət:', data['price_azn'].max())
print('Orta qiymət:', round(data['price_azn'].mean(), 2))

# 2. Vizuallaşdırma

In [ ]:
plt.figure()
plt.hist(data['price_azn'], bins=30)
plt.title('Histogram of price')
plt.xlabel('Price (AZN)')
plt.ylabel('Frequency')
plt.show()

Qiymətlərin çoxu orta hissədə toplanıb, çox baha maşınlar azdır.

In [ ]:
plt.scatter(data['mileage_km'], data['price_azn'])
plt.title('Price vs mileage')
plt.xlabel('Mileage (km)')
plt.ylabel('Price (AZN)')
plt.show()

Yürüş artdıqca qiymət ümumilikdə azalır (mənfi asılılıq).

In [ ]:
plt.scatter(data['age_years'], data['price_azn'])
plt.title('Price vs age')
plt.xlabel('Age (years)')
plt.ylabel('Price (AZN)')
plt.show()

Maşın köhnəldikcə qiyməti düşür - yaş qiymətə ən güclü təsir edən əlamətlərdən biridir.

In [ ]:
brand_mean = data.groupby('brand')['price_azn'].mean().sort_values(ascending=False)

plt.bar(brand_mean.index, brand_mean.values, color='skyblue')
plt.title('Average price by brand')
plt.xlabel('Brand')
plt.ylabel('Average price (AZN)')
plt.show()

Orta qiymətə görə BMW digər markalardan bahadır, qalan markalar bir-birinə yaxındır.

# 3. Verilənlərin hazırlanması

In [ ]:
target_col = 'price_azn'

data_fe = data.copy()
# orta illik yurus
data_fe['mileage_per_year'] = data_fe['mileage_km'] / data_fe['age_years']

numeric_cols = ['age_years', 'mileage_km', 'engine_l', 'power_hp', 'mileage_per_year']
categorical_cols = ['brand', 'fuel_type', 'transmission', 'accident_history']

In [ ]:
X = data_fe.drop(columns=[target_col])
y = data_fe[target_col]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
len(X_train), len(X_test)

## Factorize və One-Hot Encoding (brand)

In [ ]:
demo = X_train.copy()

codes, uniques = pd.factorize(demo['brand'])
demo['brand_factorized'] = codes
print(list(uniques))
demo[['brand', 'brand_factorized']].head(10)

In [ ]:
pd.get_dummies(demo['brand'], prefix='brand').head(10)

## MinMaxScaler və StandardScaler (mileage_km)
1. MinMaxScaler
$$
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
$$
2. StandardScaler
$$
x' = \frac{x - \mu}{\sigma}
$$

In [ ]:
mileage = demo[['mileage_km']].dropna()

mms = MinMaxScaler()
std = StandardScaler()

scaled = pd.DataFrame({
    'mileage_km': mileage['mileage_km'].values,
    'minmax': mms.fit_transform(mileage).ravel(),
    'standard': std.fit_transform(mileage).ravel()
})
scaled.head(10)

In [ ]:
scaled.describe().round(3)

**Fərqlər:**
- `factorize` hər markaya bir rəqəm verir (0, 1, 2, 3) - bir sütun, amma saxta sıra yaranır. OHE isə hər marka üçün ayrı 0/1 sütun yaradır - sıra yoxdur, xətti model üçün daha düzgündür.
- `MinMaxScaler` dəyərləri [0, 1] aralığına salır. `StandardScaler` ortanı 0, standart kənarlaşmanı 1 edir, dəyərlər mənfi də ola bilər. MinMax outlier-lərə daha həssasdır.

## Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# 4. Modelin öyrədilməsi və qiymətləndirilməsi

In [ ]:
linreg_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# fit yalniz train datada - imputer, encoder, scaler da train-e gore oyrenir
linreg_model.fit(X_train, y_train)
y_pred = linreg_model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'''
MAE: {mae:.2f}
MSE: {mse:.2f}
RMSE: {rmse:.2f}
R2: {r2:.2f}
''')

**Şərh:**
- **MAE** - orta hesabla proqnoz real qiymətdən bu qədər **AZN** fərqlənir.
- **MSE** - vahidi **AZN²**-dir, ona görə birbaşa şərh etmək çətindir, amma böyük xətaları göstərir.
- **RMSE** - yenə **AZN** ilədir. RMSE MAE-dən böyükdürsə deməli bəzi maşınlarda böyük xətalar var.
- **R²** - model qiymətin dəyişkənliyinin neçə faizini izah edir. 1-ə nə qədər yaxındırsa o qədər yaxşıdır.

In [ ]:
result = pd.DataFrame({
    'real_price': y_test.values[:10],
    'predicted': y_pred[:10].round(2),
})
result['abs_error'] = (result['real_price'] - result['predicted']).abs().round(2)
result

# 5. Öyrənmə əyrisi (Learning Curve)
Yalnız train datasından istifadə edirəm, test sonda qiymətləndirmə üçündür.

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    linreg_model,
    X_train,
    y_train,
    cv=5,
    scoring='r2',
    train_sizes=np.linspace(.1, 1.0, 5),
    shuffle=True,
    random_state=42)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure()
plt.plot(train_sizes, train_mean, label='Training R2', marker='o')
plt.plot(train_sizes, test_mean, 'o-', label='CV R2')
plt.title('Learning Curve (Linear Regression)')
plt.ylabel('R2 score')
plt.xlabel('Training set size')
plt.legend(loc='best')
plt.show()

**Nəticə:** Az nümunə olanda train R² yüksək, CV R² isə aşağı olur - model az datanı əzbərləyir. Nümunə artdıqca iki xətt bir-birinə yaxınlaşır. Sonda aralarındakı fərq kiçikdirsə güclü overfitting yoxdur. Xətti model sadə olduğu üçün overfitting-ə az meyllidir.

# 6. Yeni avtomobillər üçün proqnoz

In [ ]:
new_cars = pd.DataFrame({
    'age_years': [5, 10, 3],
    'mileage_km': [75000, 180000, 40000],
    'engine_l': [1.6, 2.0, 2.5],
    'power_hp': [110, 140, 170],
    'brand': ['Toyota', 'Hyundai', 'BMW'],
    'fuel_type': ['petrol', 'diesel', 'petrol'],
    'transmission': ['automatic', 'manual', 'automatic'],
    'accident_history': ['no', 'yes', 'no']
})
new_cars['mileage_per_year'] = new_cars['mileage_km'] / new_cars['age_years']

# pipeline eyni cevirmeleri ozu tetbiq edir
new_cars['predicted_price'] = linreg_model.predict(new_cars).round(2)
new_cars